In [ ]:
!pip install numpy==1.26.4 -q
!pip install diffusers transformers accelerate anthropic pillow gradio -q
!pip install git+https://github.com/facebookresearch/sam2.git -q
!pip install simple-lama-inpainting easyocr ultralytics -q
!pip install torchao peft --upgrade -q
!wget -q "https://github.com/google/fonts/raw/main/ofl/bangers/Bangers-Regular.ttf" -O /content/Bangers.ttf
print('✓ Done! Restart runtime now.')

In [ ]:
import os
import torch
import numpy as np
from PIL import Image as PILImage, ImageDraw, ImageFont
from huggingface_hub import hf_hub_download
from sam2.build_sam import build_sam2
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator
from simple_lama_inpainting import SimpleLama
from diffusers import StableDiffusionPipeline
import easyocr
import gradio as gr
from google.colab import userdata
print('✓ Імпорти OK!')

In [ ]:
try:
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except:
    os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'
    print('Встав свій API key!')

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# SD 1.5 + LoRA
MODEL_ID = 'runwayml/stable-diffusion-v1-5'
LORA_PATH = 'pytorch_lora_weights.safetensors'

print('Loading SD 1.5...')
sd_pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    safety_checker=None
).to(DEVICE)

if os.path.exists(LORA_PATH):
    sd_pipe.load_lora_weights('.')
    print(' LoRA завантажено')
else:
    print('LoRA файл не знайдено — генерація без LoRA')

# SAM2
print('Loading SAM2...')
checkpoint_path = hf_hub_download('facebook/sam2-hiera-base-plus', 'sam2_hiera_base_plus.pt')
sam2_model = build_sam2('sam2_hiera_b+.yaml', checkpoint_path, device=DEVICE)
mask_generator = SAM2AutomaticMaskGenerator(sam2_model)

# LaMa + EasyOCR
print('Loading LaMa + EasyOCR...')
lama = SimpleLama()
reader = easyocr.Reader(['en'])


In [ ]:
# BubbleCleaner 

def find_bubble_mask(image_np, masks):
    """Знаходить speech bubble серед SAM2 сегментів"""
    total_area = image_np.shape[0] * image_np.shape[1]
    candidates = []

    for mask in masks:
        seg = mask['segmentation']
        area = seg.sum()

        if area < total_area * 0.02 or area > total_area * 0.4:
            continue

        masked_pixels = image_np[seg].astype(float)
        brightness = masked_pixels.mean()
        std = masked_pixels.std()

        rows, cols = np.where(seg)
        h = rows.max() - rows.min()
        w = cols.max() - cols.min()
        aspect = min(h, w) / max(h, w) if max(h, w) > 0 else 0

        y1, y2 = int(rows.min()), int(rows.max())
        x1, x2 = int(cols.min()), int(cols.max())
        crop = image_np[y1:y2, x1:x2]
        ocr_results = reader.readtext(crop)
        has_text = len(ocr_results) > 0
        text_bonus = 200 if has_text else 0

        if brightness > 120 and std < 90:
            score = brightness * 0.4 + (255 - std) * 0.3 + aspect * 100 * 0.1 + text_bonus
            candidates.append((seg, score))

    if not candidates:
        return None

    candidates.sort(key=lambda x: x[1], reverse=True)
    return candidates[0][0]


def draw_centered_text(draw, text, cx, cy, font, fill='black'):
    
    lines = text.split('\n')
    line_height = font.getbbox('A')[3] + 8
    total_h = line_height * len(lines)
    y = cy - total_h // 2
    for line in lines:
        bbox = draw.textbbox((0, 0), line, font=font)
        lw = bbox[2] - bbox[0]
        draw.text((cx - lw // 2, y), line, fill=fill, font=font)
        y += line_height


def clean_bubble(image, text):
    if isinstance(image, np.ndarray):
        image = PILImage.fromarray(image)
    image = image.convert('RGB')
    image_np = np.array(image)

    masks = mask_generator.generate(image_np)
    bubble_mask = find_bubble_mask(image_np, masks)

    if bubble_mask is None:
        print('  Бабл не знайдено — повертаємо оригінал')
        return image


    mask_orig = np.array(
        PILImage.fromarray(bubble_mask.astype(np.uint8) * 255).resize(
            (image_np.shape[1], image_np.shape[0]), PILImage.NEAREST)
    ).astype(bool)
    bubble_pixels = image_np[mask_orig]
    bubble_color = tuple(np.percentile(bubble_pixels, 90, axis=0).astype(int))

    # LaMa inpainting
    mask_for_lama = PILImage.fromarray(bubble_mask.astype(np.uint8) * 255).resize(
        (image_np.shape[1], image_np.shape[0]), PILImage.NEAREST)
    result = lama(image, mask_for_lama)
    result_np = np.array(result).copy()

    mask_bool = np.array(
        PILImage.fromarray(bubble_mask.astype(np.uint8) * 255).resize(
            (result_np.shape[1], result_np.shape[0]), PILImage.NEAREST)
    ).astype(bool)

    result_np[mask_bool] = bubble_color
    result_final = PILImage.fromarray(result_np)
    draw = ImageDraw.Draw(result_final)

    rows, cols = np.where(mask_bool)
    bubble_h = rows.max() - rows.min()
    font_size = max(24, min(52, bubble_h // 5))
    font = ImageFont.truetype('/content/Bangers.ttf', font_size)

    row_cutoff = int(rows.min() + (rows.max() - rows.min()) * 0.55)
    upper_mask = mask_bool.copy()
    upper_mask[row_cutoff:, :] = False
    upper_rows, upper_cols = np.where(upper_mask)
    cx = int(upper_cols.mean())
    cy = int(upper_rows.mean())

    draw_centered_text(draw, text, cx, cy, font)
    return result_final


print('✓ BubbleCleaner функції визначено!')

In [ ]:
#  Pipeline 
import anthropic

client = anthropic.Anthropic()

def story_to_panels(story, num_panels=4):
    """Розбиває історію на панелі через Claude"""
    message = client.messages.create(
        model='claude-sonnet-4-20250514',
        max_tokens=1024,
        messages=[{
            'role': 'user',
            'content': f"""Split this story into exactly {num_panels} comic panels.
For each panel provide:
1. image_prompt: a detailed visual description for image generation (comicstyle, american comic book style)
2. bubble_text: short speech bubble text (max 2 lines)

Story: {story}

Respond ONLY with JSON array like:
[{{"image_prompt": "...", "bubble_text": "..."}}, ...]"""
        }]
    )
    import json
    text = message.content[0].text

    text = text.strip().strip('```json').strip('```').strip()
    return json.loads(text)


def generate_panel_image(prompt, seed=42):
    """Generate 1 panel"""
    generator = torch.Generator(DEVICE).manual_seed(seed)
    image = sd_pipe(
        prompt,
        num_inference_steps=50,
        guidance_scale=7.5,
        generator=generator
    ).images[0]
    return image


def run_full_pipeline(story, mode='adaptive', decay=0.5, seed=42,
                      char1_name='', char1_desc='', char2_name='', char2_desc=''):
    """ pipeline: story → panels → images → bubble cleaning"""
    if not story.strip():
        return None

    # Character overrides
    overrides = {}
    if char1_name.strip() and char1_desc.strip():
        overrides[char1_name.strip()] = char1_desc.strip()
    if char2_name.strip() and char2_desc.strip():
        overrides[char2_name.strip()] = char2_desc.strip()

    print('Generating panel descriptions...')
    panels = story_to_panels(story)

    result_images = []
    for i, panel in enumerate(panels):
        print(f'Panel {i+1}/{len(panels)}...')

        prompt = panel['image_prompt']
        bubble_text = panel['bubble_text']

        for name, desc in overrides.items():
            prompt = prompt.replace(name, desc)

        panel_seed = seed + i if mode == 'adaptive' else seed
        image = generate_panel_image(prompt, seed=panel_seed)

        cleaned = clean_bubble(image, bubble_text)
        result_images.append(cleaned)

    w, h = result_images[0].size
    strip = PILImage.new('RGB', (w * len(result_images), h), 'white')
    for i, img in enumerate(result_images):
        strip.paste(img, (i * w, 0))

    strip.save('/content/comic_output.png')
    return result_images + [strip]


print(' Pipeline функції визначено!')

In [ ]:
#  Gradio UI 

custom_css = """
.gradio-container .block span,
.gradio-container label span,
.gradio-container .gr-form-label,
.gradio-container .block-title,
.gradio-container .char-container h3 {
    font-size: 20px !important;
    font-weight: bold !important;
    color: white !important;
}
.gradio-container textarea,
.gradio-container input {
    font-size: 18px !important;
}
textarea::-webkit-scrollbar, input::-webkit-scrollbar { display: none !important; }
textarea, input { scrollbar-width: none !important; resize: none !important; }
input::-webkit-outer-spin-button,
input::-webkit-inner-spin-button { -webkit-appearance: none !important; margin: 0 !important; }
input[type=number] { -moz-appearance: textfield !important; }
::-webkit-resizer { display: none !important; }
input[type="range"] { accent-color: #ff8c00 !important; cursor: pointer !important; }
.buttons-row { display: flex !important; gap: 10px !important; }
.buttons-row > button { flex: 1 1 0% !important; height: 50px !important; font-size: 20px !important; font-weight: bold !important; }
.orange-btn { background-color: #ff8c00 !important; border: none !important; color: black !important; font-weight: bold !important; font-size: 20px !important; }
.orange-btn:hover { background-color: #cc7000 !important; }
.char-section-header { font-size: 24px !important; font-weight: bold !important; margin-top: 40px !important; color: white !important; }
.char-container { padding: 15px !important; border: 1px solid #444 !important; border-radius: 10px !important; margin-bottom: 15px !important; }
.equal-size { display: flex !important; align-items: flex-end !important; }
.equal-size > div { flex: 1 1 0% !important; }
.main-title { text-align: center; font-size: 36px !important; font-weight: 900 !important; color: white !important; margin-bottom: 25px !important; }
"""

js_func = """
function() {
    const textareas = document.querySelectorAll('textarea');
    textareas.forEach(el => {
        el.style.resize = 'none';
        el.style.overflow = 'hidden';
        el.addEventListener('input', function() {
            this.style.height = 'auto';
            this.style.height = (this.scrollHeight) + 'px';
        });
    });
}
"""

with gr.Blocks(css=custom_css, theme=gr.themes.Default()) as demo:

    gr.HTML("<div class='main-title'>✨TURN YOUR STORY INTO A COMIC✨</div>")

    story_input = gr.Textbox(
        label='Story',
        placeholder='Enter your story here (3-5 sentences)...',
        lines=6
    )

    with gr.Row(elem_classes=['equal-size']):
        mode_dropdown = gr.Dropdown(
            label='Mode:',
            choices=['adaptive', 'standard', 'baseline'],
            value='adaptive',
            scale=1
        )
        decay_slider = gr.Slider(
            label='Decay:',
            minimum=0.0, maximum=1.0, value=0.5, step=0.1,
            scale=2, elem_classes=['decay-number-field']
        )
        seed_input = gr.Number(label='Seed:', value=42, precision=0, scale=1)

    gr.HTML("<div class='char-section-header'>Character overrides (optional):</div>")

    with gr.Column(elem_classes=['char-container']):
        gr.Markdown('### Character 1')
        with gr.Row():
            char1_name = gr.Textbox(label='Name', placeholder="ім'я", scale=1, lines=1)
            char1_desc = gr.Textbox(label='Description', placeholder='опис', scale=3, lines=1)

    with gr.Column(elem_classes=['char-container']):
        gr.Markdown('### Character 2')
        with gr.Row():
            char2_name = gr.Textbox(label='Name', placeholder="ім'я", scale=1, lines=1)
            char2_desc = gr.Textbox(label='Description', placeholder='опис', scale=3, lines=1)

    with gr.Row(elem_classes=['buttons-row']):
        generate_btn = gr.Button('Generate Comic', variant='primary', elem_classes=['orange-btn'])

    output_gallery = gr.Gallery(show_label=False, columns=[1], object_fit='contain', height='auto')

    demo.load(None, None, None, js=js_func)

    generate_btn.click(
        fn=run_full_pipeline,
        inputs=[
            story_input, mode_dropdown, decay_slider, seed_input,
            char1_name, char1_desc, char2_name, char2_desc
        ],
        outputs=output_gallery
    )

demo.launch(debug=True, share=True)